In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import optuna
from optuna.integration import TFKerasPruningCallback

# Check for GPU availability
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
if tf.config.list_physical_devices('GPU'):
    print("GPU will be used for training")
else:
    print("WARNING: No GPU found, using CPU for training (this will be much slower)")

# Load the dataset
print("Loading dataset...")
df = pd.read_csv('bus_eta_standard_scaled.csv')
print("Data shape:", df.shape)

# Prepare the data
X = df.drop(columns=['timestamp', 'eta_minutes', 'stop_sequence'])  # Features
y = df['eta_minutes']  # Target

# Ensure all data is numeric
print("\n=== Data types before conversion ===")
print(X.dtypes)

# Convert all data to float32
X = X.astype(np.float32)
y = y.astype(np.float32)

# Check for missing values
print("\n=== Missing values ===")
print("Missing values in X:", np.isnan(X).sum())
print("Missing values in y:", np.isnan(y).sum())

# Handle missing values (fill with 0 or mean)
X = np.nan_to_num(X, nan=0.0)
y = np.nan_to_num(y, nan=0.0)

# Splitting the dataset into training, validation, and testing sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Reshape data for LSTM
def reshape_for_lstm(X, timesteps=1):
    if isinstance(X, pd.DataFrame):
        X_values = X.values.astype(np.float32)
    else:
        X_values = X.astype(np.float32)
    return np.reshape(X_values, (X_values.shape[0], timesteps, X_values.shape[1]))

timesteps = 1  # Adjust this based on your data
X_train_lstm = reshape_for_lstm(X_train, timesteps)
X_val_lstm = reshape_for_lstm(X_val, timesteps)
X_test_lstm = reshape_for_lstm(X_test, timesteps)

# Print reshaped data shapes
print("\n=== Reshaped LSTM input dimensions ===")
print(f"X_train_lstm shape: {X_train_lstm.shape}")
print(f"X_val_lstm shape: {X_val_lstm.shape}")
print(f"X_test_lstm shape: {X_test_lstm.shape}")

# Function to evaluate and visualize model performance
def evaluate_model(model, X_test, y_test, model_name):
    # Make predictions
    y_pred = model.predict(X_test, verbose=0)
    y_pred = y_pred.flatten()  # Flatten predictions for proper comparison
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    accuracy = 100 - mape
    
    print(f"--- {model_name} Performance Metrics ---")
    print(f'RMSE: {rmse:.2f}')
    print(f'R² Score: {r2:.4f}')
    print(f"MAPE: {mape:.2f}%")
    print(f"Prediction Accuracy: {accuracy:.2f}%")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    
    # Scatter plot of actual vs predicted
    axes[0, 0].scatter(y_test, y_pred, alpha=0.5)
    axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r', linestyle='--')
    axes[0, 0].set_xlabel("Actual ETA")
    axes[0, 0].set_ylabel("Predicted ETA")
    axes[0, 0].set_title(f"{model_name}: Actual vs. Predicted ETA")
    
    # Line plot of first 50 samples
    axes[0, 1].plot(y_test[:50], label="Actual", marker='o')
    axes[0, 1].plot(y_pred[:50], label="Predicted", marker='x')
    axes[0, 1].set_xlabel("Sample Index")
    axes[0, 1].set_ylabel("ETA (minutes)")
    axes[0, 1].set_title(f"{model_name}: Actual vs. Predicted ETA (First 50 Samples)")
    axes[0, 1].legend()
    
    # Histogram of errors
    errors = y_pred - y_test
    axes[1, 0].hist(errors, bins=30, edgecolor='black')
    axes[1, 0].set_xlabel("Prediction Error (Predicted - Actual)")
    axes[1, 0].set_ylabel("Frequency")
    axes[1, 0].set_title(f"{model_name}: Histogram of Prediction Errors")
    
    # Percentage error vs actual
    percentage_error = 100 * (y_pred - y_test) / y_test
    axes[1, 1].scatter(y_test, percentage_error, alpha=0.5)
    axes[1, 1].axhline(y=0, color='r', linestyle='--')
    axes[1, 1].set_xlabel("Actual ETA (minutes)")
    axes[1, 1].set_ylabel("Prediction Error (%)")
    axes[1, 1].set_title(f"{model_name}: Prediction Error vs. Actual ETA")
    
    plt.tight_layout()
    plt.show()
    
    return {
        'rmse': rmse,
        'r2': r2,
        'mape': mape,
        'accuracy': accuracy,
        'predictions': y_pred
    }

# ============== BASELINE LSTM MODEL ==============

print("\n=== Building Baseline LSTM Model ===")

def create_baseline_lstm_model(input_shape):
    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)  # Output layer for regression
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Create baseline model
input_shape = (X_train_lstm.shape[1], X_train_lstm.shape[2])
baseline_model = create_baseline_lstm_model(input_shape)

# Print model summary
print("\nBaseline LSTM Model Architecture:")
baseline_model.summary()

# Setup callbacks for training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        verbose=1,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        verbose=1,
        min_lr=1e-6
    ),
    ModelCheckpoint(
        'baseline_lstm_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train the baseline model
print("\n=== Training Baseline LSTM Model ===")
start_time = time.time()

history = baseline_model.fit(
    X_train_lstm,
    y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=50,  # We'll use early stopping to prevent overfitting
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"Baseline model training completed in {training_time:.2f} seconds")

# Save the training history
history_df = pd.DataFrame(history.history)
history_df.to_csv('baseline_lstm_training_history.csv', index=False)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error over epochs')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate the baseline model
print("\n=== Evaluating Baseline LSTM Model ===")
baseline_results = evaluate_model(baseline_model, X_test_lstm, y_test, "Baseline LSTM Model")

# ============== OPTUNA HYPERPARAMETER OPTIMIZATION ==============

def create_lstm_model(trial, input_shape):
    # Define the hyperparameter search space
    n_layers = trial.suggest_int('n_layers', 1, 3)
    
    model = Sequential()
    
    # Input layer
    lstm_units = trial.suggest_int(f'lstm_units_0', 16, 128)
    dropout_rate = trial.suggest_float(f'dropout_0', 0.0, 0.5)
    
    if n_layers == 1:
        model.add(LSTM(lstm_units, input_shape=input_shape))
        model.add(Dropout(dropout_rate))
    else:
        model.add(LSTM(lstm_units, input_shape=input_shape, return_sequences=True))
        model.add(Dropout(dropout_rate))
    
    # Hidden layers
    for i in range(1, n_layers):
        lstm_units = trial.suggest_int(f'lstm_units_{i}', 16, 128)
        dropout_rate = trial.suggest_float(f'dropout_{i}', 0.0, 0.5)
        
        if i == n_layers - 1:
            model.add(LSTM(lstm_units))
        else:
            model.add(LSTM(lstm_units, return_sequences=True))
        
        model.add(Dropout(dropout_rate))
    
    # Additional dense layer before output
    use_dense = trial.suggest_categorical('use_dense', [True, False])
    if use_dense:
        dense_units = trial.suggest_int('dense_units', 8, 64)
        model.add(Dense(dense_units, activation='relu'))
    
    # Output layer
    model.add(Dense(1))  # Single output for regression
    
    # Compile model
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    
    return model

def objective(trial):
    # Clear backend session for each trial to avoid memory growth
    tf.keras.backend.clear_session()
    
    # Generate hyperparameters
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    
    # Create model
    model = create_lstm_model(trial, input_shape)
    
    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        verbose=0,
        restore_best_weights=True
    )
    
    # Optuna pruning callback
    pruning_callback = TFKerasPruningCallback(trial, 'val_loss')
    
    # Learning rate scheduler
    lr_scheduler = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        verbose=0,
        min_lr=1e-6
    )
    
    # Train the model
    history = model.fit(
        X_train_lstm,
        y_train,
        validation_data=(X_val_lstm, y_val),
        epochs=50,  # Using early stopping
        batch_size=batch_size,
        callbacks=[early_stopping, pruning_callback, lr_scheduler],
        verbose=0
    )
    
    # Get the best validation loss
    val_loss = min(history.history['val_loss'])
    
    return val_loss

print("\n=== Starting Optuna Hyperparameter Optimization ===")
print("This process may take some time depending on your GPU...")
start_time = time.time()

# Create and run the study
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=30)  # Adjust n_trials as needed based on time constraints

optimization_time = time.time() - start_time
print(f"Optimization completed in {optimization_time:.2f} seconds")

# Get the best parameters
best_params = study.best_params
print("\n=== Best Hyperparameters ===")
for param_name, param_value in best_params.items():
    print(f"{param_name}: {param_value}")

# ============== TRAINING FINAL OPTIMIZED MODEL ==============

print("\n=== Training Final Optimized LSTM Model ===")
start_time = time.time()

# Clear backend session
tf.keras.backend.clear_session()

# Create model with the best parameters
final_model = create_lstm_model(study.best_trial, input_shape)

# Setup callbacks for final training
final_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        verbose=1,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        verbose=1,
        min_lr=1e-6
    ),
    ModelCheckpoint(
        'optimized_lstm_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train the final model
final_history = final_model.fit(
    X_train_lstm,
    y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=100,  # Using early stopping
    batch_size=best_params['batch_size'],
    callbacks=final_callbacks,
    verbose=1
)

final_training_time = time.time() - start_time
print(f"Final model training completed in {final_training_time:.2f} seconds")

# Save the training history
final_history_df = pd.DataFrame(final_history.history)
final_history_df.to_csv('optimized_lstm_training_history.csv', index=False)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(final_history.history['loss'], label='Training Loss')
plt.plot(final_history.history['val_loss'], label='Validation Loss')
plt.title('Loss over epochs (Optimized Model)')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(final_history.history['mae'], label='Training MAE')
plt.plot(final_history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error over epochs (Optimized Model)')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate the final model
print("\n=== Evaluating Optimized LSTM Model ===")
optimized_results = evaluate_model(final_model, X_test_lstm, y_test, "Optuna Optimized LSTM Model")

# ============== COMPARE BASELINE AND OPTIMIZED MODELS ==============

print("\n=== Model Performance Comparison ===")
print(f"{'Metric':<15} {'Baseline LSTM':<15} {'Optimized LSTM':<15}")
print("-" * 45)
print(f"{'RMSE':<15} {baseline_results['rmse']:<15.2f} {optimized_results['rmse']:<15.2f}")
print(f"{'R² Score':<15} {baseline_results['r2']:<15.4f} {optimized_results['r2']:<15.4f}")
print(f"{'MAPE':<15} {baseline_results['mape']:<15.2f}% {optimized_results['mape']:<15.2f}%")
print(f"{'Accuracy':<15} {baseline_results['accuracy']:<15.2f}% {optimized_results['accuracy']:<15.2f}%")

# Save the model
final_model.save('optimized_lstm_model.h5')
print("\nOptimized LSTM model saved as 'optimized_lstm_model.h5'")

# ============== TIME HORIZON PREDICTIONS ==============

# Function to filter test data for specific time horizons
def get_horizon_data(X_test, y_test, horizon, tolerance=0.5):
    """
    Filter the test data to include samples close to the specified time horizon.
    """
    # Convert to numpy arrays if they're not already
    if isinstance(X_test, pd.DataFrame):
        X_test = X_test.values
    if isinstance(y_test, pd.Series):
        y_test = y_test.values
    
    # Find indices where y_test is within the horizon ± tolerance
    horizon_mask = (y_test >= horizon - tolerance) & (y_test <= horizon + tolerance)
    
    if not horizon_mask.any():
        print(f"No exact data points found for horizon {horizon} minutes (±{tolerance})")
        # If no exact matches, get closest values
        distances = np.abs(y_test - horizon)
        closest_indices = np.argsort(distances)[:max(int(len(y_test) * 0.05), 10)]
        horizon_mask = np.zeros_like(y_test, dtype=bool)
        horizon_mask[closest_indices] = True
        print(f"Using {sum(horizon_mask)} closest data points instead")
    else:
        print(f"Found {sum(horizon_mask)} data points for horizon {horizon} minutes (±{tolerance})")
    
    # For LSTM, we need to reshape the data
    if X_test.ndim == 2:  # If X_test is not already shaped for LSTM
        X_horizon = X_test[horizon_mask]
        X_horizon = np.reshape(X_horizon, (X_horizon.shape[0], 1, X_horizon.shape[1]))
    else:  # If X_test is already shaped for LSTM
        X_horizon = X_test[horizon_mask]
    
    y_horizon = y_test[horizon_mask]
    
    return X_horizon, y_horizon

# Define the time horizons in minutes
time_horizons = [1, 3, 5, 15, 30, 60]

# Results storage
horizon_results = {}

# Test the model on each time horizon
print("\n=== Time Horizon Performance Comparison ===")
print(f"{'Time Horizon':<15} {'Samples':<10} {'RMSE':<10} {'R²':<10} {'MAPE':<10} {'Accuracy':<10}")
print("-" * 65)

for horizon in time_horizons:
    # Get data for this horizon
    X_horizon, y_horizon = get_horizon_data(X_test_lstm, y_test, horizon)
    
    if len(X_horizon) < 5:  # Skip if too few samples
        print(f"{horizon} min: Insufficient data")
        continue
    
    # Make predictions
    y_pred = final_model.predict(X_horizon, verbose=0).flatten()
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_horizon, y_pred))
    r2 = r2_score(y_horizon, y_pred)
    
    # Avoid division by zero in MAPE calculation
    valid_indices = y_horizon != 0
    if np.any(valid_indices):
        mape = np.mean(np.abs((y_horizon[valid_indices] - y_pred[valid_indices]) / y_horizon[valid_indices])) * 100
    else:
        mape = np.nan
    
    accuracy = 100 - mape if not np.isnan(mape) else np.nan
    
    horizon_results[horizon] = {
        'samples': len(X_horizon),
        'rmse': rmse,
        'r2': r2,
        'mape': mape,
        'accuracy': accuracy
    }
    
    print(f"{horizon} min:{'':<9} {len(X_horizon):<10d} {rmse:<10.2f} {r2:<10.4f} {mape:<10.2f}% {accuracy:<10.2f}%")

# Visualization of accuracy across time horizons
plt.figure(figsize=(12, 6))
horizons = list(horizon_results.keys())
accuracies = [horizon_results[h]['accuracy'] for h in horizons]
rmse_values = [horizon_results[h]['rmse'] for h in horizons]

fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = 'tab:blue'
ax1.set_xlabel('Time Horizon (minutes)')
ax1.set_ylabel('Accuracy (%)', color=color1)
ax1.plot(horizons, accuracies, marker='o', color=color1, linestyle='-', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('RMSE', color=color2)
ax2.plot(horizons, rmse_values, marker='s', color=color2, linestyle='--', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('LSTM Model Performance Across Different Time Horizons')
plt.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Save horizon results
horizon_df = pd.DataFrame.from_dict(horizon_results, orient='index')
horizon_df.index.name = 'horizon_minutes'
horizon_df.to_csv('lstm_time_horizon_results.csv')
print("\nTime horizon results saved to 'lstm_time_horizon_results.csv'")

print("\n=== LSTM Model Implementation Complete ===")

c:\Users\cheng\anaconda3\envs\GPU\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow version: 2.1.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU will be used for training
Loading dataset...
Data shape: (100000, 14)

=== Data types before conversion ===
current_stop_name          int64
next_stop_name             int64
day_of_week                int64
is_holiday                  bool
is_peak_hour                bool
weather_condition          int64
passenger_count          float64
current_speed            float64
distance_to_next_stop    float64
current_lat              float64
current_lon              float64
dtype: object

=== Missing values ===
Missing values in X: current_stop_name        0
next_stop_name           0
day_of_week              0
is_holiday               0
is_peak_hour             0
weather_condition        0
passenger_count          0
current_speed            0
distance_to_next_stop    0
current_lat              0
current_lon              0
dtype: int64
Missing values in y: 0

=== Reshaped LSTM input 